In [1]:
from moviepy import VideoFileClip

video = VideoFileClip("interview.mp4")
video.audio.write_audiofile("audio.wav")

MoviePy - Writing audio in audio.wav


MoviePy - Done.


In [2]:
import whisper

model = whisper.load_model("base")

result = model.transcribe(r"D:\University\Fourth Level\Second Term\GP\audio.wav", word_timestamps=True)


words = []
for seg in result["segments"]:
    for w in seg["words"]:
        words.append(w)


c:\Python312\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [4]:
duration = words[-1]['end'] - words[0]['start']
total_words = len(words)

wpm = total_words / (duration/60)

print("WPM:", wpm)

WPM: 210.0238663484487


In [6]:
PAUSE_THRESHOLD = 0.8
pauses = []

for i in range(len(words)-1):
    gap = words[i+1]['start'] - words[i]['end']
    if gap > PAUSE_THRESHOLD:   
        pauses.append(gap)

pause_count = len(pauses)
pause_ratio = sum(pauses) / duration


In [7]:
fillers = ["um", "uh", "like", "you know", "ah"]

filler_count = 0

for w in words:
    if w["word"].lower() in fillers:
        filler_count += 1

print("Filler Words:", filler_count)

Filler Words: 0


In [8]:
speech_score = 1 - abs(wpm - 160) / 160
speech_score = max(0, min(speech_score, 1))


pause_score = 1 - pause_ratio
pause_score = max(0, min(pause_score, 1))


smooth_score = 1 - (filler_count / total_words)
smooth_score = max(0, min(smooth_score, 1))

In [9]:
fluency = (
    0.4 * speech_score +
    0.3 * pause_score +
    0.3 * smooth_score
)

fluency_score = fluency * 100

print("Speech Score:", speech_score)
print("Pause Score:", pause_score)
print("Smoothness Score:", smooth_score)

print("Final Fluency Score:", fluency_score)

Speech Score: 0.6873508353221955
Pause Score: 1.0
Smoothness Score: 1.0
Final Fluency Score: 87.49403341288782
